# Taller 2 — Titanic: análisis exploratorio de datos (EDA)

**Pregunta central:** ¿qué factores tuvieron más influencia sobre la supervivencia de los pasajeros del Titanic?

Este cuaderno es de **exploración**: lee Bronze y Silver, produce gráficos y tablas, y deja la lógica productiva en `flows/`.
Las decisiones y su interpretación se resumen en `reports/conclusiones_eda.md`.

> Regla: si una celda calcula algo que otro proceso necesitará, ese cálculo vive en `flows/`, no aquí.


## 0. Preparación del entorno

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from flows.common.paths import BRONZE, BRONZE_LABELED_NAME, BRONZE_QUALITY_NAME, GOLD, SILVER

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.titlesize'] = 13
print('pandas', pd.__version__, '| seaborn', sns.__version__)

## Parte 1 — Primer contacto con los datos (Pasos 1–5)

Se empieza por **Bronze**, la copia sin retocar, para no ocultar problemas con la limpieza.

In [ ]:
# 1) Cargamos Bronze (nunca la URL: la ingesta ya la guardó)
df = pd.read_csv(BRONZE / BRONZE_LABELED_NAME)

### Paso 1. ¿Cuántos pasajeros y variables hay?

In [ ]:
df.shape

### Paso 2. ¿Cuáles son las variables, sus nombres y tipos?

In [ ]:
df.info()

### Paso 3. Primeras 10 filas

In [ ]:
df.head(10)

### Paso 4. Estadísticas descriptivas

In [ ]:
df.describe()

In [ ]:
df.select_dtypes(exclude='number').describe()

### Paso 5. Datos faltantes: tabla, mapa de calor y reporte automático

Los nulos no son un detalle: **cómo se rellenen cambia las conclusiones**. Aquí solo los diagnosticamos.


In [ ]:
df.isnull().sum()

In [ ]:
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Mapa de datos faltantes (Bronze)')
plt.show()

In [ ]:
quality = json.loads((BRONZE / BRONZE_QUALITY_NAME).read_text(encoding='utf-8'))
pd.Series(quality['null_pct']).round(1).sort_values(ascending=False)

**Lectura.** De 891 pasajeros, a `Age` le faltan 177 (19,9 %), a `Cabin` 687 (77,1 %) y a `Embarked` 2 (0,2 %).
El mapa de calor no muestra una fila entera vacía: la pérdida está concentrada en columnas concretas.


## Parte 2 — Conociendo a los pasajeros (Pasos 6–10)

A partir de aquí se lee **Silver**: las columnas derivadas ya fueron calculadas por `flows/01_data_engineering.py`.


In [ ]:
df = pd.read_parquet(SILVER / 'titanic_silver.parquet')
df.shape

### Paso 6. Distribución por género y su cruce con la clase

In [ ]:
sns.countplot(data=df, x='Sex')
plt.title('Pasajeros por género')
plt.show()

In [ ]:
sns.countplot(data=df, x='Sex', hue='Pclass')
plt.title('Género por clase de tiquete')
plt.show()

In [ ]:
pd.crosstab(df['Sex'], df['Pclass'], normalize='columns').round(3)

### Paso 7. Distribución de edades

In [ ]:
sns.histplot(df['Age'].dropna(), bins=30)
plt.title('Distribución de la edad (solo edades conocidas)')
plt.xlabel('Edad')
plt.show()

In [ ]:
print('Media:', round(df['AgeOriginal'].mean(), 2))
print('Mediana:', df['AgeOriginal'].median())
print('Mínimo:', df['AgeOriginal'].min(), '| Máximo:', df['AgeOriginal'].max())

### Paso 8. Variable `Person` (hombre, mujer o niño)

Regla explícita: `Age < 16` → `child`; si no, se conserva `Sex`. Un pasajero **sin edad nunca es niño**
(`NaN < 16` es `False` en NumPy): se queda con su sexo. Ese sesgo se cuantifica más adelante.


In [ ]:
df['Person'].value_counts()

**Reto 8.1.** De los 177 pasajeros sin edad, muchos podrían ser niños. Si la proporción de niños
entre los que sí tienen edad (83 de 714 ≈ 11,6 %) se sostuviera, faltarían ~21 `child`. La tasa observada
de supervivencia infantil podría estar subestimada o diluida en las categorías `male`/`female`.


### Paso 9. Niños por clase

In [ ]:
pd.crosstab(df['Pclass'], df['Person'])

### Paso 10. Densidades (KDE) de la edad

`common_norm=False` normaliza cada curva por separado: así un grupo grande no "aplasta" a uno pequeño.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
sns.kdeplot(data=df, x='Age', hue='Sex', fill=True, common_norm=False, ax=axes[0])
axes[0].set_title('Edad por género')
sns.kdeplot(data=df, x='Age', hue='Person', fill=True, common_norm=False, ax=axes[1])
axes[1].set_title('Edad por tipo de persona')
sns.kdeplot(data=df, x='Age', hue='Pclass', fill=True, common_norm=False,
            palette='viridis', ax=axes[2])
axes[2].set_title('Edad por clase')
plt.tight_layout()
plt.show()

## Parte 3 — Ubicación, embarque y composición familiar (Pasos 11–15)

### Paso 11. Cubierta de la cabina

`TCabin` es la primera letra de `Cabin`; `X` significa "desconocida" (una categoría explícita, no un borrado).
`HasCabin` marca si había cabina registrada. `Cabin` original se conserva en Silver.


In [ ]:
df['TCabin'].value_counts()

### Paso 12. Conteo por cubierta (excluyendo lo desconocido y lo anómalo del gráfico)

In [ ]:
cubierta = df.loc[df['TCabin'] != 'X', 'TCabin']
sns.countplot(x=cubierta, order=sorted(cubierta.unique()))
plt.title('Pasajeros con cabina registrada, por cubierta')
plt.show()

In [ ]:
cubierta = cubierta[cubierta != 'T']   # 1 solo registro: se excluye del GRÁFICO, no de df
sns.countplot(x=cubierta, order=sorted(cubierta.unique()))
plt.title('Cubiertas (sin la cubierta T anómala)')
plt.xlabel('Cubierta')
plt.show()

### Paso 13. Puerto de embarque y clase

In [ ]:
sns.countplot(data=df, x='Embarked', hue='Pclass')
plt.title('Puerto de embarque por clase')
plt.show()

In [ ]:
pd.crosstab(df['Embarked'], df['Pclass'], normalize='index').round(3)

### Paso 14–15. Variable `Alone`

In [ ]:
df['Alone'].value_counts(normalize=True).round(3)

In [ ]:
sns.countplot(data=df, x='Alone')
plt.title('Viajaba solo o con familia')
plt.show()

**Lectura.** 537 pasajeros (60,3 %) viajaban solos y 354 (39,7 %) con familia.
La composición por clase difiere mucho entre puertos: Queenstown es casi toda de 3.ª clase,
mientras Cherbourg tiene ~51 % de 1.ª. Eso será una **variable de confusión** en la Parte 9.


## Parte 4 — Análisis de supervivencia: el núcleo del EDA (Pasos 16–20)

La media de `Survived` (0/1) es la **tasa de supervivencia**. Las barras verticales de los gráficos de puntos
son intervalos de confianza al 95 %: miden incertidumbre, no importancia.


In [ ]:
sns.countplot(data=df, x='Survivor')
plt.title('Supervivencia (etiqueta de texto)')
plt.show()
print('Tasa general:', round(df['Survived'].mean(), 4))

### Paso 17. Supervivencia por clase

In [ ]:
g = sns.catplot(data=df, x='Pclass', y='Survived', kind='point', height=4, aspect=1.4)
g.ax.set_title('Tasa de supervivencia por clase')
g.ax.set_ylabel('Tasa de supervivencia')
plt.show()

### Paso 18. Supervivencia por clase y tipo de persona

In [ ]:
g = sns.catplot(data=df, x='Pclass', y='Survived', hue='Person',
                hue_order=['child', 'female', 'male'], kind='point', height=4, aspect=1.5)
g.ax.set_title('Supervivencia por clase y tipo de persona')
g.ax.set_ylabel('Tasa de supervivencia')
plt.show()

### Paso 19. Edad y supervivencia

In [ ]:
sns.lmplot(data=df, x='Age', y='Survived')
plt.title('Tendencia lineal edad vs supervivencia')
plt.show()

In [ ]:
sns.lmplot(data=df, x='Age', y='Survived', hue='Pclass')
plt.title('Edad vs supervivencia, por clase')
plt.show()

In [ ]:
generaciones = [10, 20, 40, 60, 80]
sns.lmplot(data=df, x='Age', y='Survived', hue='Pclass', x_bins=generaciones)
plt.title('Tasa por intervalo de edad y clase')
plt.show()

In [ ]:
sns.lmplot(data=df, x='Age', y='Survived', hue='Sex', x_bins=generaciones)
plt.title('Tasa por intervalo de edad y género')
plt.show()

### Paso 20. Relación tarifa–edad

In [ ]:
plt.scatter(df['Age'], df['Fare'], alpha=0.5)
plt.title('Tarifa vs edad')
plt.xlabel('Edad')
plt.ylabel('Tarifa')
plt.show()

In [ ]:
sns.scatterplot(data=df, x='Age', y='Fare', hue='Pclass', alpha=0.5, palette='viridis')
plt.title('Tarifa vs edad, coloreado por clase')
plt.show()

## Parte 5 — Limpieza y preparación (Pasos 21–26)

### Paso 21. `Embarked`: moda y codificación

In [ ]:
print('Faltantes en Bronze:', 2)
print('Moda:', df['Embarked'].mode()[0])
df[['Embarked', 'NEmbarked']].drop_duplicates().sort_values('Embarked')

### Paso 22. `Age`: imputación por género y su costo

In [ ]:
print('Desv. estándar con edad original:', round(df['AgeOriginal'].std(), 2))
print('Desv. estándar tras imputar:      ', round(df['Age'].std(), 2))
print('Filas imputadas:', int(df['AgeImputada'].sum()))

In [ ]:
sns.histplot(df['Age'].dropna(), bins=30, color='C1')
plt.title('Edad ya imputada: el pico central son valores inventados')
plt.show()

**Reto 22.1.** La desviación baja de 14,53 a 13,01 porque 177 valores se sustituyeron por promedios
de grupo: se reduce la variabilidad artificialmente y el histograma desarrolla un pico central que no
es datos reales. Para **explorar** está bien si se documenta; para **modelar** se vuelve a imputar dentro
del `Pipeline`, usando solo estadísticas de entrenamiento.


### Pasos 23–24. Codificaciones `NSex` y `NCabin`

In [ ]:
df[['Sex', 'NSex']].drop_duplicates()

In [ ]:
df['NCabin'].value_counts().sort_index()

El mapa de cubiertas es **fijo** (`DECKS = ABCDEFGTX`), no depende de qué categorías aparezcan en el
momento: así los resultados siguen siendo comparables entre ejecuciones. `X` (desconocida) queda como la
última categoría; el orden es una simplificación para correlaciones, no para modelar.


### Pasos 25–26. Títulos del nombre y su consolidación

In [ ]:
df['TTitulo'].value_counts()

In [ ]:
pd.crosstab(df['TTitulo'], df['Sex'])

In [ ]:
df['NTitulo'].value_counts().sort_index()

**Cuestionamiento.** `Master` (niño varón) se fusionó con `Mr`. Esa decisión mezcla niños con adultos;
la compensa parcialmente `Person`, que usa la edad original. Es una simplificación defendible solo si se
documenta, como aquí.


## Parte 6 — Correlaciones y estadísticas por grupos (Pasos 27–29)

In [ ]:
cols = ['Survived', 'Pclass', 'NSex', 'Age', 'SibSp', 'Parch', 'Fare',
        'NEmbarked', 'NCabin', 'NTitulo']
corr = df[cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Matriz de correlación')
plt.show()
corr['Survived'].sort_values()

**Recordatorio.** Correlación lineal ≠ causalidad, y las codificaciones `N…` imponen un orden
arbitrario: el signo de `NCabin` no tiene interpretación sustantiva.


### Paso 28. Estadísticas por grupos (tablas Gold)

In [ ]:
for name in ['age_by_survival', 'survival_by_sex', 'survival_by_pclass', 'survival_by_alone']:
    print('\n---', name, '---')
    display(pd.read_parquet(GOLD / f'{name}.parquet').round(3))

### Paso 29. ¿Es la diferencia por sexo estadísticamente significativa?

In [ ]:
chi2 = json.loads((GOLD / 'chi2_sex_survival.json').read_text(encoding='utf-8'))
chi2

**Conclusión con límites.** χ² ≈ 260,7, gl = 1, p ≈ 1,2e-58: rechazamos la independencia entre
sexo y supervivencia. Esto dice que la asociación es muy improbable que sea azar; **no** dice que sea grande,
ni que el sexo *causara* la supervivencia, ni que el mismo patrón deba operar hoy.


## Parte 7 — Visualizaciones comparativas avanzadas (Pasos 30–31)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, col in zip(axes.flat, ['Sex', 'Pclass', 'Embarked', 'Survived']):
    sns.boxplot(data=df, x=col, y='AgeOriginal', ax=ax)   # edades REALES, no imputadas
    ax.set_title(f'Edad real por {col}')
plt.tight_layout()
plt.show()

In [ ]:
sns.histplot(df['Fare'], kde=True, bins=40)
plt.title('Distribución de la tarifa')
plt.show()
print('Media:', round(df['Fare'].mean(), 2), '| Mediana:', round(df['Fare'].median(), 2),
      '| Asimetría:', round(df['Fare'].skew(), 2))

In [ ]:
sns.histplot(np.log1p(df['Fare']), kde=True, bins=30)
plt.title('log(1 + tarifa): reduce la asimetría')
plt.show()

La tarifa es muy asimétrica (skew ≈ 4,8): la media (32,2) está lejos de la mediana (14,5).
Un modelo lineal que use `Fare` sin transformar puede verse dominado por unas pocas tarifas extremas.


## Parte 8 — Dataset final y verificación (Pasos 32–33)

In [ ]:
final = pd.read_parquet(SILVER / 'titanic_final.parquet')
print('Silver completo:', df.shape, '| vista final:', final.shape)
print('Columnas eliminadas en la copia:', {'Name', 'Sex', 'Embarked', 'Cabin'} & set(df.columns))
final.isnull().sum().sort_values(ascending=False).head()

In [ ]:
final.dtypes

`AgeOriginal` conserva sus NaN a propósito: es la verdad observada. `validate_silver` exige sin nulos
en las columnas críticas, pero explícitamente permite nulos en `AgeOriginal`.


## Parte 9 — Análisis crítico: interacciones, asociaciones y atípicos (Pasos 34–38)

### Paso 34. Clase × género

In [ ]:
tabla = pd.read_parquet(GOLD / 'survival_by_class_sex.parquet')
display(tabla.round(3))
sns.heatmap(tabla.set_index('Pclass'), annot=True, fmt='.2f', cmap='YlGnBu')
plt.title('Tasa de supervivencia: clase × género')
plt.show()

Las mujeres de 1.ª y 2.ª clase sobreviven en ~97 % y ~92 % (n = 144 y 93); los hombres de 3.ª,
solo ~13,5 % (n = 347). La interacción importa: el efecto del género no es igual en todas las clases.


### Paso 35. Puerto de embarque: ¿confusión con la clase?

In [ ]:
display(pd.read_parquet(GOLD / 'survival_by_embarked.parquet').round(3))
display(pd.read_parquet(GOLD / 'survival_by_embarked_class.parquet').round(3))
sns.pointplot(data=df, x='Embarked', y='Survived', hue='Pclass')
plt.title('Supervivencia por puerto y clase')
plt.show()

En bruto, Cherbourg parece "mejor" (55,4 %) que Southampton (33,9 %). Al estratificar por clase,
la ventaja se reduce dentro de cada clase (p. ej. en 3.ª: 37,9 % vs 19,0 % sigue habiendo diferencia, pero
parte del efecto bruto venía de que Cherbourg traía más pasajeros de 1.ª). Estratificar separa la confusión.


### Paso 36. Tarifa dentro de cada clase

In [ ]:
# El cuartil se calcula DENTRO de cada clase (de lo contrario sería un proxy de Pclass).
# La tabla Gold ya lo aplica en flows/03_aggregation.py; aquí lo replicamos para graficar.
df_con_q = df.copy()
df_con_q['FareQ'] = df_con_q.groupby('Pclass')['Fare'].transform(
    lambda s: pd.qcut(s, 4, labels=False, duplicates='drop'))

In [ ]:
display(pd.read_parquet(GOLD / 'survival_by_fare_quartile_class.parquet').round(3))
sns.pointplot(data=df_con_q, x='FareQ', y='Survived', hue='Pclass', errorbar='ci')
plt.title('Supervivencia por cuartil de tarifa y clase')
plt.show()

Dentro de 1.ª clase, pagar más en el propio cuartil se asocia con tasas de 46 % → 57 % → 75 % → 74 %.
Pero `Fare` y `Pclass` están fuertemente correlacionadas: sin estratificar, la tarifa sería un proxy de clase.


### Paso 37. Tamaño de la familia

In [ ]:
familia = pd.read_parquet(GOLD / 'survival_by_family.parquet')
display(familia.round(3))
sns.pointplot(data=df, x='Familia', y='Survived')
plt.title('Supervivencia según familiares a bordo')
plt.show()

Familias pequeñas (1–3 familiares: 55 %–72 %, n = 161/102/29) sobreviven más que quienes viajan solos
(30 %, n = 537) o con 5+ (≈ 14 % o 0 %, n = 22/6/7). Las barras de error se disparan en familias grandes:
son grupos de menos de 20 personas, así que no conviene afirmar un "número óptimo" con precisión.


### Paso 38. Atípicos (regla IQR)

In [ ]:
display(pd.read_parquet(GOLD / 'outlier_summary.parquet').round(3))
df.nlargest(10, 'Fare')[['Name', 'Pclass', 'Fare', 'Survived']]

116 tarifas (13 %) son atípicas por IQR y sobrevivió el 68,1 % de ellas frente al 33,9 % del resto.
Pero son en su mayoría pasajeros de 1.ª clase: **atípico no significa erróneo ni protector**. Eliminarlos
sin inspeccionarlos borraría justo al grupo con más supervivencia.


## Conclusión del EDA

1. **Género y clase** son las asociaciones más fuertes: mujeres 74,2 % (n = 314) vs hombres 18,9 % (n = 577);
   1.ª clase 63,0 % (n = 216) vs 3.ª 24,2 % (n = 491).
2. **"Mujeres y niños primero"** se cumple en términos generales, con excepciones: en 3.ª clase la tasa
   femenina cae a 50 % y los niños de 1.ª tienen intervalos muy anchos por falta de muestra.
3. **Familia pequeña** se asocia con mayor supervivencia; las familias grandes tienen muestras mínimas.
4. **Datos faltantes y decisiones de limpieza importan**: 177 edades imputadas por género reducen la desviación
   de 14,5 a 13,0; 687 cabinas desconocidas se convirtieron en `X`; `Master` se fusionó con `Mr`.
5. **Lo que NO podemos afirmar**: asociación no es causalidad; es un solo naufragio; la información de cabina
   pudo registrarse de forma distinta para supervivientes; y los subgrupos pequeños no permiten generalizar.

La formulación del problema predictivo y sus límites están en `reports/problema_ml.md`.
